# Overmind Quickstart

This notebook demonstrates how to use the Overmind Python SDK as a drop-in replacement for OpenAI, Anthropic, and Google Gemini clients. Every LLM call is automatically traced and sent to the Overmind platform for optimization.

We'll run the same mock customer-support agent across all three providers — 30 queries each, 90 traced calls total. The system prompt is **intentionally suboptimal**: it's vague, contradictory, and encourages hallucination. This is on purpose — after enough traces, Overmind's optimization pipeline will detect the issues and suggest concrete improvements.

**Sections:**
1. [Setup](#1-setup)
2. [Traced LLM calls](#2-traced-llm-calls)
   - [2.1 OpenAI](#21-openai)
   - [2.2 Anthropic](#22-anthropic)
   - [2.3 Google Gemini](#23-google-gemini)
3. [What happens next — the Overmind optimization pipeline](#3-what-happens-next)

## 1. Setup

Install the SDK and the provider libraries you want to use.

In [ ]:
%pip install --quiet overmind openai anthropic google-genai

Set your API keys. You need an Overmind key plus at least one provider key.

Get a free Overmind API key at [console.overmindlab.ai](https://console.overmindlab.ai), or use a self-hosted instance ([github.com/overmind-core/overmind](https://github.com/overmind-core/overmind)).

In [ ]:
import os

os.environ["OVERMIND_API_KEY"] = "<your-overmind-key>"

# You need at least one provider key
os.environ["OPENAI_API_KEY"] = "<your-openai-key>"
os.environ["ANTHROPIC_API_KEY"] = "<your-anthropic-key>"
os.environ["GEMINI_API_KEY"] = "<your-gemini-key>"

### System prompt & customer queries

The system prompt below is **deliberately bad** — it's wordy, self-contradictory, and vague about what TechCorp actually does. It also tells the model to guess when it doesn't know the answer. This is the kind of prompt Overmind's optimization pipeline is designed to catch and improve.

In [ ]:
SYSTEM_PROMPT = (
    "You are a customer support assistant for TechCorp, a software company. "
    "You should try to help customers but also be friendly and professional at the same time. "
    "Make sure your responses are not too long but also contain enough information to be helpful. "
    "If someone asks about something you don't know, try your best to give an answer anyway "
    "because customers expect a response and an empty reply is never acceptable under any circumstances. "
    "You can suggest they contact support via email if needed but try not to do that too often "
    "because it's not a great customer experience when you do that. "
    "Be empathetic but also be efficient with your responses and don't waste time. "
    "Try to resolve things in one message if possible but also don't rush the customer. "
    "TechCorp makes various software products and technology solutions for businesses and consumers. "
    "Always greet the customer warmly and ask if there's anything else you can help with at the end. "
    "Remember that customer satisfaction is the number one priority but also keep in mind that "
    "you should follow company policies at all times. If there's a conflict between what the "
    "customer wants and what the policy says, use your best judgment to figure it out. "
    "Try to sound natural and human-like but also be precise and accurate. "
    "Don't use too many exclamation marks but be enthusiastic. "
    "You may offer discounts if the customer seems upset but only if it feels right."
)

CUSTOMER_QUERIES = [
    "How do I reset my password?",
    "I was charged twice for my subscription.",
    "Can I get a refund for my last order?",
    "My account was locked — what do I do?",
    "How do I upgrade to the premium plan?",
    "I need to cancel my subscription.",
    "The app keeps crashing on my phone.",
    "Where can I find my invoice?",
    "How do I change my email address on file?",
    "I'm not receiving any notification emails.",
    "What are your business hours?",
    "Can I transfer my license to another person?",
    "How long does shipping take for physical products?",
    "I received the wrong item in my order.",
    "Do you offer student discounts?",
    "How do I enable two-factor authentication?",
    "My promo code isn't working at checkout.",
    "Can I pause my subscription instead of canceling?",
    "How do I export my data from the platform?",
    "I forgot which email I used to sign up.",
    "Is there an API available for developers?",
    "How do I add another user to my team account?",
    "The website shows an error when I try to checkout.",
    "Can I change my subscription billing date?",
    "I need a copy of my contract.",
    "How do I delete my account permanently?",
    "What payment methods do you accept?",
    "I'm having trouble connecting third-party integrations.",
    "Can you explain the difference between your plans?",
    "I need to update my billing address.",
]

print(f"System prompt length: {len(SYSTEM_PROMPT)} chars")
print(f"Customer queries: {len(CUSTOMER_QUERIES)}")

---

## 2. Traced LLM Calls

Each section below creates an Overmind-wrapped client, sends all 30 queries, and prints each response. The only thing that changes between providers is the import and the call signature — everything else is identical.

### 2.1 OpenAI

Swap `from openai import OpenAI` → `from overmind.clients import OpenAI`. That's it.

In [ ]:
from overmind.clients import OpenAI

openai_client = OpenAI()

for i, query in enumerate(CUSTOMER_QUERIES, 1):
    response = openai_client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": query},
        ],
    )
    print(f"[{i:02d}] Q: {query}")
    print(f"     A: {response.choices[0].message.content}\n")

### 2.2 Anthropic

Swap `from anthropic import Anthropic` → `from overmind.clients import Anthropic`.

In [ ]:
from overmind.clients import Anthropic

anthropic_client = Anthropic()

for i, query in enumerate(CUSTOMER_QUERIES, 1):
    response = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": query},
        ],
    )
    print(f"[{i:02d}] Q: {query}")
    print(f"     A: {response.content[0].text}\n")

### 2.3 Google Gemini

Use `from overmind.clients.google import Client` instead of `from google import genai`.

In [ ]:
from overmind.clients.google import Client as GoogleClient
from google.genai import types

gemini_client = GoogleClient()

for i, query in enumerate(CUSTOMER_QUERIES, 1):
    response = gemini_client.models.generate_content(
        model="gemini-3-flash-preview",
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
        ),
        contents=query,
    )
    print(f"[{i:02d}] Q: {query}")
    print(f"     A: {response.text}\n")

---

## 3. What Happens Next

You've just sent **30 traced LLM calls using one or multiple providers** to Overmind. Here's what the platform does with them:

### 3.1 Trace Collection

Every call — input, output, model, latency, token counts, cost — is recorded as a trace. These appear in your dashboard within seconds.

### 3.2 Agent Detection

After ~10 calls with a similar prompt structure, Overmind groups them into an **agent**. It extracts the underlying prompt template (system prompt + variable user message) and tracks it over time.

### 3.3 LLM-as-Judge Scoring

A judge model evaluates each trace on three dimensions:
- **Quality** — Is the response accurate and relevant?
- **Cost** — Could a cheaper model produce the same result?
- **Latency** — Is the response time acceptable?

With the deliberately vague system prompt we used, expect the judge to flag issues: hallucinated company details, inconsistent tone, unnecessarily long responses.

### 3.4 Prompt Experimentation

Overmind generates improved prompt variants and replays your inputs through them. Since our prompt is wordy and contradictory, the optimized version will likely be shorter, more specific, and free of conflicting instructions.

### 3.5 Model Experimentation

Your inputs are replayed through alternative models (e.g., `gpt-5-nano`, `claude-haiku-4-5`, `gemini-3.1-flash-lite-preview`) to find cheaper or faster options that maintain quality.

### 3.6 Actionable Suggestions

All results are distilled into concrete recommendations on your Overmind dashboard: swap prompt X for Y, switch from model A to model B, expected savings of Z%.

### 3.7 Feedback Loop

Accept or reject suggestions. Accepted changes feed back into the system so future recommendations improve.

---

Head to your [dashboard](https://console.overmindlab.ai) to see traces arriving in real time. For more details, see the [full documentation](https://docs.overmindlab.ai).